In [15]:
# NYC Taxi Data Processing with PySpark - OPTIMIZED FOR PARQUET


import os
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, avg, count, sum, min, max, hour, dayofweek, 
    month, year, round as spark_round, when, lit, date_format, to_timestamp
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd


In [16]:
# ============================================
# 1. INITIALIZE OPTIMIZED SPARK SESSION
# ============================================
print("Initializing Optimized Spark Session...")
spark = (
    SparkSession.builder
    .appName("NYCTaxiParquetOptimized")
    .config("spark.master", "local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", "128m")
    .config("spark.rdd.compress", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.parquet.mergeSchema", "false")
    .config("spark.sql.parquet.filterPushdown", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark ready - using {spark.sparkContext.defaultParallelism} cores.")


Initializing Optimized Spark Session...
Spark ready - using 4 cores.


25/11/15 20:26:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [17]:
# ============================================
# 2. DEFINE PROPER SCHEMA (CRITICAL!)
# ============================================
print("\n" + "="*50)
print("DEFINING SCHEMA")
print("="*50)

# Define schema with correct data types
taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),  # Will convert to timestamp
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("RateCodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])


DEFINING SCHEMA


In [18]:
# ============================================
# 3. LOAD AND CONVERT TO PARQUET (IF NOT ALREADY DONE)
# ============================================
print("\n" + "="*50)
print("CHECKING FOR PARQUET DATA")
print("="*50)

parquet_path = "/kaggle/working/taxi_parquet"

# Check if parquet already exists
if not os.path.exists(parquet_path):
    print("Parquet not found. Converting CSV to Parquet...")
    
    data_path = "/kaggle/input/nyc-yellow-taxi-trip-data"
    files = [
        "yellow_tripdata_2015-01.csv",
        "yellow_tripdata_2016-01.csv",
        "yellow_tripdata_2016-02.csv",
        "yellow_tripdata_2016-03.csv"
    ]
    
    # Load CSV with proper schema
    df_csv = spark.read.csv(
        [os.path.join(data_path, f) for f in files],
        header=True,
        schema=taxi_schema
    )
    
    print(f"CSV records loaded: {df_csv.count():,}")
    
    # Write to Parquet (much faster for subsequent reads)
    print("Writing to Parquet format...")
    df_csv.write.mode("overwrite").parquet(parquet_path)
    print("✅ Parquet conversion complete!")
else:
    print("✅ Parquet data already exists")



CHECKING FOR PARQUET DATA
✅ Parquet data already exists


In [19]:
# ============================================
# 4. LOAD PARQUET DATA
# ============================================
print("\n" + "="*50)
print("LOADING PARQUET DATA")
print("="*50)

df = spark.read.parquet(parquet_path)
print(f"Total records loaded: {df.count():,}")
print("\nSchema:")
df.printSchema()
print("\nSample data:")
df.show(5)


LOADING PARQUET DATA
Total records loaded: 47,248,845

Schema:
root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)


Sample data:
+--------+--------------------+------------------

In [23]:
# ============================================
# 4. LOAD PARQUET DATA & FIX DATA TYPES
# ============================================
print("\n" + "="*50)
print("LOADING PARQUET DATA")
print("="*50)

df_raw = spark.read.parquet(parquet_path)
print(f"Total records loaded: {df_raw.count():,}")
print("\n⚠️ Original Schema (may have wrong types):")
df_raw.printSchema()

# CRITICAL FIX: Cast all columns to correct types
print("\nCasting columns to correct data types...")
df = df_raw.select(
    col("VendorID").cast(IntegerType()).alias("VendorID"),
    col("tpep_pickup_datetime").cast(StringType()).alias("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime").cast(StringType()).alias("tpep_dropoff_datetime"),
    col("passenger_count").cast(IntegerType()).alias("passenger_count"),
    col("trip_distance").cast(DoubleType()).alias("trip_distance"),
    col("pickup_longitude").cast(DoubleType()).alias("pickup_longitude"),
    col("pickup_latitude").cast(DoubleType()).alias("pickup_latitude"),
    col("RateCodeID").cast(IntegerType()).alias("RateCodeID"),
    col("store_and_fwd_flag").cast(StringType()).alias("store_and_fwd_flag"),
    col("dropoff_longitude").cast(DoubleType()).alias("dropoff_longitude"),
    col("dropoff_latitude").cast(DoubleType()).alias("dropoff_latitude"),
    col("payment_type").cast(IntegerType()).alias("payment_type"),
    col("fare_amount").cast(DoubleType()).alias("fare_amount"),
    col("extra").cast(DoubleType()).alias("extra"),
    col("mta_tax").cast(DoubleType()).alias("mta_tax"),
    col("tip_amount").cast(DoubleType()).alias("tip_amount"),
    col("tolls_amount").cast(DoubleType()).alias("tolls_amount"),
    col("improvement_surcharge").cast(DoubleType()).alias("improvement_surcharge"),
    col("total_amount").cast(DoubleType()).alias("total_amount")
)

print("✅ Data types corrected!")
print("\nCorrected Schema:")
df.printSchema()
print("\nSample data:")
df.show(5)



LOADING PARQUET DATA
Total records loaded: 47,248,845

⚠️ Original Schema (may have wrong types):
root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)


Casting columns to correct 

In [24]:
# ============================================
# 5. DATA CLEANING & FEATURE ENGINEERING
# ============================================
print("\n" + "="*50)
print("CLEANING & FEATURE ENGINEERING")
print("="*50)

# Convert datetime strings to timestamps
df = df.withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime"))) \
       .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))

# Add time-based features FIRST (before filtering)
df_enriched = df \
    .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day", dayofweek("tpep_pickup_datetime")) \
    .withColumn("pickup_month", month("tpep_pickup_datetime")) \
    .withColumn("pickup_year", year("tpep_pickup_datetime")) \
    .withColumn("pickup_date", date_format("tpep_pickup_datetime", "yyyy-MM-dd")) \
    .withColumn("trip_duration_minutes", 
                (col("tpep_dropoff_datetime").cast("long") - 
                 col("tpep_pickup_datetime").cast("long")) / 60.0)

# Add derived features
df_enriched = df_enriched \
    .withColumn("speed_mph", 
                when((col("trip_duration_minutes") > 0) & (col("trip_distance") > 0),
                     col("trip_distance") / (col("trip_duration_minutes") / 60.0))
                .otherwise(0)) \
    .withColumn("tip_percentage",
                when(col("fare_amount") > 0,
                     (col("tip_amount") / col("fare_amount")) * 100.0)
                .otherwise(0))

print("Features added. Now applying filters...")

# Clean data - RELAXED filters to avoid empty results
df_clean = df_enriched.filter(
    # Basic validity checks
    (col("fare_amount").isNotNull()) & (col("fare_amount") > 0) & (col("fare_amount") < 500) &
    (col("trip_distance").isNotNull()) & (col("trip_distance") > 0) & (col("trip_distance") < 100) &
    (col("passenger_count").isNotNull()) & (col("passenger_count") > 0) & (col("passenger_count") < 9) &
    (col("total_amount").isNotNull()) & (col("total_amount") > 0) &
    
    # Datetime validity
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull()) &
    (col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") < 180) &
    
    # Coordinate validity (more lenient - some values might be 0)
    (col("pickup_latitude").isNotNull()) &
    (col("pickup_longitude").isNotNull()) &
    (col("dropoff_latitude").isNotNull()) &
    (col("dropoff_longitude").isNotNull())
)

records_after_cleaning = df_clean.count()
print(f"Records after cleaning: {records_after_cleaning:,}")

if records_after_cleaning == 0:
    print("\n⚠️ WARNING: All records filtered out!")
    print("\nDiagnostic: Checking data ranges BEFORE filtering...")
    
    # Show what the data actually looks like
    print("\nFare amount range:")
    df_enriched.select(min("fare_amount"), max("fare_amount")).show()
    
    print("\nTrip distance range:")
    df_enriched.select(min("trip_distance"), max("trip_distance")).show()
    
    print("\nPassenger count distribution:")
    df_enriched.groupBy("passenger_count").count().orderBy("passenger_count").show()
    
    print("\nCoordinate ranges:")
    df_enriched.select(
        min("pickup_latitude"), max("pickup_latitude"),
        min("pickup_longitude"), max("pickup_longitude")
    ).show()
    
    print("\nTrip duration range:")
    df_enriched.select(min("trip_duration_minutes"), max("trip_duration_minutes")).show()
    
    print("\nNull counts per column:")
    from pyspark.sql.functions import isnan, when, count as count_fn
    df_enriched.select([count_fn(when(col(c).isNull(), c)).alias(c) for c in df_enriched.columns]).show()
    
    print("\n⚠️ Using UNFILTERED data for processing to avoid empty dataset")
    df_clean = df_enriched
    
print(f"\n✅ Final dataset size: {df_clean.count():,} records")

# Cache for better performance
df_clean.cache()
print("Dataset cached in memory")



CLEANING & FEATURE ENGINEERING
Features added. Now applying filters...


Records after cleaning: 46,873,693



✅ Final dataset size: 46,873,693 records
Dataset cached in memory


In [25]:
# ============================================
# 6. SPARK SQL AGGREGATIONS
# ============================================
print("\n" + "="*50)
print("RUNNING SPARK SQL AGGREGATIONS")
print("="*50)

df_clean.createOrReplaceTempView("taxi_trips")

# Aggregation 1: Daily Statistics
print("1. Computing daily statistics...")
daily_stats = df_clean.groupBy("pickup_date", "pickup_year", "pickup_month").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration"),
    spark_round(sum("total_amount"), 2).alias("total_revenue"),
    spark_round(avg("passenger_count"), 2).alias("avg_passengers"),
    spark_round(avg("tip_percentage"), 2).alias("avg_tip_percent"),
    spark_round(max("fare_amount"), 2).alias("max_fare"),
    spark_round(min("fare_amount"), 2).alias("min_fare")
).orderBy("pickup_date")

print(f"Daily stats records: {daily_stats.count()}")

# Aggregation 2: Hourly Demand Patterns
print("2. Computing hourly demand patterns...")
hourly_demand = df_clean.groupBy("pickup_hour").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("passenger_count"), 2).alias("avg_passengers"),
    spark_round(sum("total_amount"), 2).alias("total_revenue")
).orderBy("pickup_hour")

# Aggregation 3: Day of Week Patterns
print("3. Computing day of week patterns...")
dow_stats = df_clean.groupBy("pickup_day").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance"),
    spark_round(avg("tip_percentage"), 2).alias("avg_tip_percent"),
    spark_round(sum("total_amount"), 2).alias("total_revenue")
).orderBy("pickup_day")

# Aggregation 4: Payment Type Analysis
print("4. Computing payment type statistics...")
payment_stats = df_clean.groupBy("payment_type").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("tip_amount"), 2).alias("avg_tip"),
    spark_round(avg("tip_percentage"), 2).alias("avg_tip_percent")
)

# Aggregation 5: Rate Code Analysis
print("5. Computing rate code statistics...")
ratecode_stats = df_clean.groupBy("RateCodeID").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance")
).orderBy("RateCodeID")

# Aggregation 6: Passenger Count Distribution
print("6. Computing passenger distribution...")
passenger_dist = df_clean.groupBy("passenger_count").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("tip_percentage"), 2).alias("avg_tip_percent")
).orderBy("passenger_count")

# Aggregation 7: Distance Bins
print("7. Computing distance-based statistics...")
distance_bins = df_clean.withColumn(
    "distance_category",
    when(col("trip_distance") < 2, "0-2 miles")
    .when(col("trip_distance") < 5, "2-5 miles")
    .when(col("trip_distance") < 10, "5-10 miles")
    .when(col("trip_distance") < 20, "10-20 miles")
    .otherwise("20+ miles")
)

distance_stats = distance_bins.groupBy("distance_category").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration")
)

# Aggregation 8: Month-over-Month Trends
print("8. Computing monthly trends...")
monthly_stats = df_clean.groupBy("pickup_year", "pickup_month").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(sum("total_amount"), 2).alias("total_revenue")
).orderBy("pickup_year", "pickup_month")


RUNNING SPARK SQL AGGREGATIONS
1. Computing daily statistics...


25/11/15 21:01:45 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 222.2 MiB so far)
25/11/15 21:01:45 WARN BlockManager: Persisting block rdd_156_10 to disk instead.
25/11/15 21:01:45 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 223.1 MiB so far)
25/11/15 21:01:45 WARN BlockManager: Persisting block rdd_156_8 to disk instead.
25/11/15 21:01:45 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 222.4 MiB so far)
25/11/15 21:01:45 WARN BlockManager: Persisting block rdd_156_9 to disk instead.
25/11/15 21:02:18 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 142.4 MiB so far)
25/11/15 21:02:33 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 41.5 MiB so far)
25/11/15 21:02:40 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 21.2 MiB so far)
25/11/15 21:02:40 WARN BlockManager: Persisting block rdd_156_15 to disk instead.
25/11/1

Daily stats records: 122
2. Computing hourly demand patterns...
3. Computing day of week patterns...
4. Computing payment type statistics...
5. Computing rate code statistics...
6. Computing passenger distribution...
7. Computing distance-based statistics...
8. Computing monthly trends...


In [26]:
# ============================================
# 7. MACHINE LEARNING WITH MLlib
# ============================================
print("\n" + "="*50)
print("BUILDING ML MODELS")
print("="*50)

# Prepare features for ML (filter out nulls)
ml_df = df_clean.filter(
    col("pickup_latitude").isNotNull() &
    col("pickup_longitude").isNotNull() &
    col("dropoff_latitude").isNotNull() &
    col("dropoff_longitude").isNotNull()
)

feature_cols = [
    'trip_distance', 'passenger_count', 'pickup_hour', 
    'pickup_day', 'RateCodeID'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_ml = assembler.transform(ml_df)

# Sample for faster training (10% of data)
df_sample = df_ml.sample(fraction=0.1, seed=42)
train_data, test_data = df_sample.randomSplit([0.8, 0.2], seed=42)

print(f"Training set size: {train_data.count():,}")
print(f"Test set size: {test_data.count():,}")

# MODEL 1: Random Forest - Fare Prediction
print("\n1. Training Random Forest for fare prediction...")
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=30,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# Evaluate
evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = evaluator.evaluate(rf_predictions)
r2 = evaluator.setMetricName("r2").evaluate(rf_predictions)
mae = evaluator.setMetricName("mae").evaluate(rf_predictions)

print(f"Random Forest RMSE: ${rmse:.2f}")
print(f"Random Forest R²: {r2:.4f}")
print(f"Random Forest MAE: ${mae:.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.featureImportances.toArray()
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Save sample predictions
rf_predictions_sample = rf_predictions.select(
    "fare_amount", "prediction", "trip_distance", 
    "passenger_count", "pickup_hour", "pickup_day"
).limit(10000)

# MODEL 2: K-Means Clustering - Popular Routes
print("\n2. Training K-Means for route clustering...")

# Only use records with valid coordinates
route_df = df_clean.filter(
    (col("pickup_latitude") != 0) & (col("pickup_longitude") != 0) &
    (col("dropoff_latitude") != 0) & (col("dropoff_longitude") != 0)
).sample(fraction=0.05, seed=42)

route_features = ['pickup_latitude', 'pickup_longitude', 
                  'dropoff_latitude', 'dropoff_longitude']

route_assembler = VectorAssembler(inputCols=route_features, outputCol="route_features", handleInvalid="skip")
df_routes = route_assembler.transform(route_df)

kmeans = KMeans(k=10, featuresCol="route_features", seed=42, maxIter=20)
kmeans_model = kmeans.fit(df_routes)

# Get cluster centers
centers = kmeans_model.clusterCenters()
cluster_df = pd.DataFrame(centers, columns=route_features)
cluster_df['cluster_id'] = range(len(centers))

print(f"Cluster centers computed: {len(centers)}")

# Assign clusters and analyze
clustered = kmeans_model.transform(df_routes)
cluster_stats = clustered.groupBy("prediction").agg(
    count("*").alias("trip_count"),
    spark_round(avg("fare_amount"), 2).alias("avg_fare"),
    spark_round(avg("trip_distance"), 2).alias("avg_distance")
).withColumnRenamed("prediction", "cluster_id")


BUILDING ML MODELS


25/11/15 21:08:17 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 231.5 MiB so far)
25/11/15 21:08:29 WARN MemoryStore: Not enough space to cache rdd_156_5 in memory! (computed 223.4 MiB so far)
25/11/15 21:08:36 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 142.3 MiB so far)
25/11/15 21:08:36 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 142.0 MiB so far)
25/11/15 21:08:46 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 141.4 MiB so far)
25/11/15 21:08:46 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 371.3 MiB so far)
25/11/15 21:08:47 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 140.9 MiB so far)
25/11/15 21:08:48 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 141.1 MiB so far)


Training set size: 3,749,144


25/11/15 21:09:04 WARN MemoryStore: Not enough space to cache rdd_156_0 in memory! (computed 292.2 MiB so far)
25/11/15 21:09:15 WARN MemoryStore: Not enough space to cache rdd_156_4 in memory! (computed 231.5 MiB so far)
25/11/15 21:09:23 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 223.1 MiB so far)
25/11/15 21:09:23 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 142.0 MiB so far)
25/11/15 21:09:24 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 222.4 MiB so far)
25/11/15 21:09:33 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 221.3 MiB so far)
25/11/15 21:09:35 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 141.1 MiB so far)
25/11/15 21:09:36 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 370.5 MiB so far)


Test set size: 936,700

1. Training Random Forest for fare prediction...


25/11/15 21:09:56 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 231.5 MiB so far)
25/11/15 21:10:06 WARN MemoryStore: Not enough space to cache rdd_156_6 in memory! (computed 223.4 MiB so far)
25/11/15 21:10:15 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 223.1 MiB so far)
25/11/15 21:10:15 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 41.5 MiB so far)
25/11/15 21:10:25 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 221.3 MiB so far)
25/11/15 21:10:25 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 371.3 MiB so far)
25/11/15 21:10:27 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 370.5 MiB so far)
25/11/15 21:10:27 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 221.1 MiB so far)
25/11/15 21:10:42 WARN MemoryStore: Not enough space to cache rdd_156_0 in memory! (computed 292.2 MiB so fa

Random Forest RMSE: $2.82
Random Forest R²: 0.9235
Random Forest MAE: $1.53

Feature Importance:
           feature  importance
0    trip_distance    0.664736
4       RateCodeID    0.327957
2      pickup_hour    0.004955
3       pickup_day    0.001699
1  passenger_count    0.000653

2. Training K-Means for route clustering...


25/11/15 21:17:21 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 85.0 MiB so far)
25/11/15 21:17:22 WARN MemoryStore: Not enough space to cache rdd_156_3 in memory! (computed 147.8 MiB so far)
25/11/15 21:17:22 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 147.8 MiB so far)
25/11/15 21:17:24 WARN MemoryStore: Not enough space to cache rdd_156_4 in memory! (computed 231.5 MiB so far)
25/11/15 21:17:25 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 142.0 MiB so far)
25/11/15 21:17:25 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 222.4 MiB so far)
25/11/15 21:17:26 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 21.3 MiB so far)
25/11/15 21:17:28 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 371.3 MiB so far)
25/11/15 21:17:30 WARN MemoryStore: Not enough space to cache rdd_156_3 in memory! (computed 85.0 MiB so far)
2

Cluster centers computed: 10


In [27]:
# ============================================
# 8. SAVE RESULTS TO JSON
# ============================================
print("\n" + "="*50)
print("SAVING RESULTS")
print("="*50)

output_dir = "/kaggle/working"

# Helper function to safely convert and save
def save_to_json(df, filename):
    try:
        pdf = df.toPandas()
        filepath = f"{output_dir}/{filename}"
        pdf.to_json(filepath, orient='records', indent=2)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"✅ {filename} ({size_kb:.2f} KB)")
    except Exception as e:
        print(f"❌ Error saving {filename}: {e}")

print("\nSaving aggregations:")
save_to_json(daily_stats, "daily_stats.json")
save_to_json(hourly_demand, "hourly_demand.json")
save_to_json(dow_stats, "dow_stats.json")
save_to_json(payment_stats, "payment_stats.json")
save_to_json(ratecode_stats, "ratecode_stats.json")
save_to_json(passenger_dist, "passenger_dist.json")
save_to_json(distance_stats, "distance_stats.json")
save_to_json(monthly_stats, "monthly_stats.json")

print("\nSaving ML results:")
save_to_json(rf_predictions_sample, "fare_predictions.json")
save_to_json(cluster_stats, "cluster_stats.json")

print("\nSaving metadata:")
cluster_df.to_json(f"{output_dir}/cluster_centers.json", orient='records', indent=2)
feature_importance.to_json(f"{output_dir}/feature_importance.json", orient='records', indent=2)

# Save comprehensive model metrics
model_metrics = {
    "random_forest": {
        "rmse": float(rmse),
        "r2": float(r2),
        "mae": float(mae),
        "num_trees": 30,
        "max_depth": 10
    },
    "kmeans": {
        "num_clusters": 10,
        "total_trips_analyzed": int(df_routes.count())
    },
    "data_summary": {
        "total_records_raw": int(df.count()),
        "total_records_cleaned": int(df_clean.count()),
        "cleaning_retention_rate": f"{(df_clean.count() / df.count() * 100):.2f}%",
        "date_range_start": str(daily_stats.agg(min('pickup_date')).collect()[0][0]),
        "date_range_end": str(daily_stats.agg(max('pickup_date')).collect()[0][0])
    },
    "feature_importance": feature_importance.to_dict('records')
}

with open(f"{output_dir}/model_metrics.json", 'w') as f:
    json.dump(model_metrics, f, indent=2)

print("✅ model_metrics.json")


SAVING RESULTS

Saving aggregations:


25/11/15 21:27:25 WARN MemoryStore: Not enough space to cache rdd_156_3 in memory! (computed 85.0 MiB so far)
25/11/15 21:27:26 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 85.0 MiB so far)
25/11/15 21:27:29 WARN MemoryStore: Not enough space to cache rdd_156_7 in memory! (computed 41.7 MiB so far)
25/11/15 21:27:30 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 41.6 MiB so far)
25/11/15 21:27:32 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 142.0 MiB so far)
25/11/15 21:27:32 WARN MemoryStore: Not enough space to cache rdd_156_11 in memory! (computed 141.6 MiB so far)
25/11/15 21:27:35 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 371.3 MiB so far)
25/11/15 21:27:36 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 141.1 MiB so far)
25/11/15 21:27:36 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 220.6 MiB so far)


✅ daily_stats.json (38.32 KB)


25/11/15 21:27:40 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 231.5 MiB so far)
25/11/15 21:27:42 WARN MemoryStore: Not enough space to cache rdd_156_7 in memory! (computed 142.2 MiB so far)
25/11/15 21:27:42 WARN MemoryStore: Not enough space to cache rdd_156_9 in memory! (computed 142.3 MiB so far)
25/11/15 21:27:43 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 142.0 MiB so far)
25/11/15 21:27:43 WARN MemoryStore: Not enough space to cache rdd_156_11 in memory! (computed 141.6 MiB so far)
25/11/15 21:27:44 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 220.8 MiB so far)
25/11/15 21:27:45 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_156_15 in memory.
25/11/15 21:27:45 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 384.0 B so far)
25/11/15 21:27:46 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (c

✅ hourly_demand.json (3.97 KB)


...

✅ fare_predictions.json (1509.19 KB)


25/11/15 21:28:42 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 147.8 MiB so far)
25/11/15 21:28:42 WARN MemoryStore: Not enough space to cache rdd_156_3 in memory! (computed 147.8 MiB so far)
25/11/15 21:28:42 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 147.8 MiB so far)
25/11/15 21:28:46 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 142.4 MiB so far)
25/11/15 21:28:48 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 81.1 MiB so far)
25/11/15 21:28:48 WARN MemoryStore: Not enough space to cache rdd_156_11 in memory! (computed 221.8 MiB so far)
25/11/15 21:28:48 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 140.9 MiB so far)
25/11/15 21:28:50 WARN MemoryStore: Not enough space to cache rdd_156_16 in memory! (computed 220.4 MiB so far)


✅ cluster_stats.json (0.95 KB)

Saving metadata:


25/11/15 21:28:52 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 85.0 MiB so far)
25/11/15 21:28:52 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 147.8 MiB so far)
25/11/15 21:28:52 WARN MemoryStore: Not enough space to cache rdd_156_3 in memory! (computed 147.8 MiB so far)
25/11/15 21:28:55 WARN MemoryStore: Not enough space to cache rdd_156_11 in memory! (computed 141.6 MiB so far)
25/11/15 21:28:55 WARN MemoryStore: Not enough space to cache rdd_156_8 in memory! (computed 223.1 MiB so far)
25/11/15 21:28:55 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 21.3 MiB so far)
25/11/15 21:28:56 WARN MemoryStore: Not enough space to cache rdd_156_16 in memory! (computed 21.2 MiB so far)
25/11/15 21:28:57 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 370.5 MiB so far)
25/11/15 21:28:58 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 85.0 MiB so far)
2

✅ model_metrics.json


In [28]:
# ============================================
# 9. SUMMARY STATISTICS
# ============================================
print("\n" + "="*50)
print("PROCESSING COMPLETE!")
print("="*50)

print("\nAll output files:")
for filename in sorted(os.listdir(output_dir)):
    if filename.endswith('.json'):
        filepath = os.path.join(output_dir, filename)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  📄 {filename:30s} {size_kb:>10.2f} KB")

print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)
print(f"Total trips processed: {df_clean.count():,}")
print(f"Data cleaning retention: {model_metrics['data_summary']['cleaning_retention_rate']}")
print(f"Date range: {model_metrics['data_summary']['date_range_start']} to {model_metrics['data_summary']['date_range_end']}")

avg_fare_result = daily_stats.agg(avg('avg_fare')).collect()[0][0]
avg_dist_result = daily_stats.agg(avg('avg_distance')).collect()[0][0]

print(f"Average fare: ${avg_fare_result:.2f}")
print(f"Average trip distance: {avg_dist_result:.2f} miles")
print(f"\nModel Performance:")
print(f"  Fare Prediction RMSE: ${rmse:.2f}")
print(f"  Fare Prediction R²: {r2:.4f}")
print(f"  Fare Prediction MAE: ${mae:.2f}")
print(f"\nRoute Clusters Identified: {len(centers)}")

print("\n✅ All JSON files ready in /kaggle/working/ - Download them for your dashboard!")

# Stop Spark
spark.stop()
print("\n🛑 Spark session stopped")



PROCESSING COMPLETE!

All output files:
  📄 cluster_centers.json                 1.77 KB
  📄 cluster_stats.json                   0.95 KB
  📄 daily_stats.json                    38.32 KB
  📄 distance_stats.json                  0.58 KB
  📄 dow_stats.json                       1.17 KB
  📄 fare_predictions.json             1509.19 KB
  📄 feature_importance.json              0.34 KB
  📄 hourly_demand.json                   3.97 KB
  📄 model_metrics.json                   0.89 KB
  📄 monthly_stats.json                   0.57 KB
  📄 passenger_dist.json                  0.85 KB
  📄 payment_stats.json                   0.60 KB
  📄 ratecode_stats.json                  0.68 KB

SUMMARY STATISTICS


25/11/15 21:36:46 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 85.0 MiB so far)
25/11/15 21:36:46 WARN MemoryStore: Not enough space to cache rdd_156_5 in memory! (computed 81.8 MiB so far)
25/11/15 21:36:46 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 147.8 MiB so far)
25/11/15 21:36:47 WARN MemoryStore: Not enough space to cache rdd_156_6 in memory! (computed 41.6 MiB so far)
25/11/15 21:36:47 WARN MemoryStore: Not enough space to cache rdd_156_12 in memory! (computed 21.3 MiB so far)
25/11/15 21:36:47 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 41.5 MiB so far)
25/11/15 21:36:47 WARN MemoryStore: Not enough space to cache rdd_156_0 in memory! (computed 292.2 MiB so far)
25/11/15 21:36:48 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 221.1 MiB so far)


Total trips processed: 46,873,693
Data cleaning retention: 99.21%
Date range: 2015-01-01 to 2016-03-31


25/11/15 21:36:49 WARN MemoryStore: Not enough space to cache rdd_156_2 in memory! (computed 85.0 MiB so far)
25/11/15 21:36:49 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 85.0 MiB so far)
25/11/15 21:36:51 WARN MemoryStore: Not enough space to cache rdd_156_6 in memory! (computed 21.4 MiB so far)
25/11/15 21:36:52 WARN MemoryStore: Not enough space to cache rdd_156_7 in memory! (computed 41.7 MiB so far)
25/11/15 21:36:54 WARN MemoryStore: Not enough space to cache rdd_156_10 in memory! (computed 41.5 MiB so far)
25/11/15 21:36:56 WARN MemoryStore: Not enough space to cache rdd_156_13 in memory! (computed 81.1 MiB so far)
25/11/15 21:36:57 WARN MemoryStore: Not enough space to cache rdd_156_14 in memory! (computed 140.9 MiB so far)
25/11/15 21:36:57 WARN MemoryStore: Not enough space to cache rdd_156_15 in memory! (computed 141.1 MiB so far)
25/11/15 21:37:01 WARN MemoryStore: Not enough space to cache rdd_156_1 in memory! (computed 147.8 MiB so far)
25/

Average fare: $12.33
Average trip distance: 2.89 miles

Model Performance:
  Fare Prediction RMSE: $2.82
  Fare Prediction R²: 0.9235
  Fare Prediction MAE: $1.53

Route Clusters Identified: 10

✅ All JSON files ready in /kaggle/working/ - Download them for your dashboard!

🛑 Spark session stopped
